# Brique 0 — Structurer un projet Python comme un pro

Tu as sûrement déjà ouvert des repos GitLab chez Cozycozy et vu des dossiers `src/`, `lib/`, des fichiers
`requirements.txt`, `.env`, `.gitignore`. Ce notebook construit, un par un, le problème concret que chacun de
ces éléments résout — pas juste "voici la convention", mais "voici ce qui se passe mal si on ne l'a pas".

On avance dans cet ordre :
1. Le problème de fond : installer des bibliothèques, et pourquoi ça peut mal se passer
2. L'environnement virtuel (`venv`) : la solution à ce problème
3. `requirements.txt` : déclarer ce dont le projet a besoin
4. `src/` : pourquoi le code ne doit pas traîner directement à la racine
5. `lib/` : ce que c'est vraiment, et pourquoi tu le connais déjà en Java
6. `.gitignore` : dire à Git ce qu'il ne doit jamais toucher
7. `.env` : séparer les secrets du code
8. Vue d'ensemble, puis un exercice guidé pour tout mettre en pratique toi-même

## 1. Le problème de fond : qu'est-ce qui se passe quand tu "installes" une bibliothèque

Quand tu veux utiliser une bibliothèque externe en Python (par exemple `pandas`, qu'on verra en Brique 4), tu
tapes une commande comme `pip install pandas`. Il faut comprendre ce que cette commande fait réellement : elle
**télécharge le code source de pandas et le copie dans un dossier précis de ton ordinateur**, un dossier que
Python sait aller consulter automatiquement chaque fois que tu écris `import pandas`.

Par défaut, sans rien faire de particulier, ce dossier est **unique pour tout ton ordinateur**, partagé par
absolument tous tes projets Python. C'est là que le problème commence à apparaître.

**Construisons un exemple concret du problème.** Imagine que tu as deux projets :

- Le *projet A*, écrit il y a un an, qui fonctionne avec la version 1.5 de `pandas` (les bibliothèques évoluent
  dans le temps, et parfois une nouvelle version change le comportement de certaines fonctions, ou en supprime
  certaines).
- Le *projet B*, que tu démarres aujourd'hui, et qui a besoin d'une fonctionnalité qui n'existe que dans la
  version 2.1 de `pandas`.

Si `pandas` n'existe qu'en un seul exemplaire, partagé par toute la machine, **tu ne peux pas avoir les deux
versions installées en même temps**. Si tu installes la 2.1 pour le projet B, tu casses potentiellement le
projet A, qui comptait sur le comportement de la 1.5. Si tu reviens à la 1.5 pour réparer A, tu casses B.

C'est exactement le problème que doit résoudre la section suivante.

## 2. L'environnement virtuel (`venv`) : donner à chaque projet sa propre boîte à outils

**L'idée de solution, avant le mot technique.** Au lieu d'avoir un seul endroit partagé où sont installées
toutes les bibliothèques pour tous les projets, on va donner à **chaque projet son propre dossier privé**, qui
contient sa propre copie de l'interpréteur Python et ses propres bibliothèques installées, complètement séparée
des autres projets. Le projet A a sa boîte à outils avec `pandas` 1.5 dedans ; le projet B a sa propre boîte à
outils, séparée, avec `pandas` 2.1 dedans. Les deux boîtes n'ont aucune idée que l'autre existe.

Ce dossier privé, propre à un seul projet, s'appelle un **environnement virtuel** (en anglais, *virtual
environment*, souvent raccourci en `venv`, qui est aussi le nom de l'outil intégré à Python qui sert à le
créer).

**Comment on en crée un, concrètement**, dans un terminal, à la racine de ton projet :

```bash
python -m venv .venv
```

Cette commande crée un dossier appelé `.venv` (le nom est une convention, tu pourrais l'appeler autrement, mais
`.venv` est ce que tu verras partout). À l'intérieur, tu trouveras une copie de l'interpréteur Python, et un
sous-dossier qui accueillera les bibliothèques installées *pour ce projet seulement*.

**Mais ce dossier existe, il ne sert à rien tout seul.** Il faut ensuite dire à ton terminal "à partir de
maintenant, utilise cette boîte à outils-ci, pas celle du système". C'est ce qu'on appelle **activer**
l'environnement :

```bash
# Sur Windows :
.venv\Scripts\activate

# Sur macOS ou Linux :
source .venv/bin/activate
```

Une fois cette commande lancée, tu verras en général le nom de l'environnement apparaître entre parenthèses au
début de la ligne de ton terminal (par exemple `(.venv) C:\...>`). C'est le signal visuel que tu es "à
l'intérieur" de cette boîte à outils. À partir de ce moment, si tu tapes `pip install pandas`, pandas sera
installé **dans ce dossier `.venv` précis**, pas dans un endroit partagé par tout ton ordinateur.

**Ce que ça veut dire pour la suite** : ce dossier `.venv` contient potentiellement des centaines de mégaoctets
de fichiers de bibliothèques. Il n'a **aucune raison** d'être partagé avec quelqu'un d'autre ou d'être suivi par
Git — n'importe qui peut le recréer en deux commandes, à partir du fichier `requirements.txt` qu'on construit
dans la section suivante. On reverra ce point précis dans la section `.gitignore`.

## 3. `requirements.txt` : écrire noir sur blanc ce dont le projet dépend

**Le problème que ça résout.** Maintenant que chaque projet a son propre environnement virtuel, vide au départ,
il faut un moyen de dire "ce projet a besoin de telle et telle bibliothèque, à telle version", pour que n'importe
qui (ou un serveur de déploiement automatique, plus tard) puisse reconstruire exactement le même environnement
sans avoir à deviner quoi installer.

**La solution : un simple fichier texte**, nommé par convention `requirements.txt`, avec une bibliothèque par
ligne :

```
pandas==2.1.4
numpy>=1.26
scikit-learn
```

Chaque ligne peut s'écrire de trois façons différentes, et le choix a une vraie conséquence :

- `pandas==2.1.4` : fige une version **exacte**. Avantage : reproductible à 100%, le comportement du projet ne
  changera jamais tout seul. Inconvénient : il faut mettre à jour ce numéro soi-même pour profiter des
  nouveautés ou des corrections de bugs futures.
- `numpy>=1.26` : fixe seulement un minimum. Plus souple, mais une future version de numpy pourrait changer un
  comportement et casser ton code sans que tu aies touché à quoi que ce soit.
- `scikit-learn` : aucune contrainte, toujours la toute dernière version disponible au moment de l'installation.
  Pratique en apprentissage personnel ; dangereux en production, car le projet peut se comporter différemment du
  jour au lendemain sans qu'aucune ligne de code n'ait changé.

**Et pour reconstruire l'environnement à partir de ce fichier**, une fois le `venv` créé et activé :

```bash
pip install -r requirements.txt
```

`-r` veut dire "lis la liste des bibliothèques à installer depuis ce fichier", plutôt que de les écrire une par
une à la main dans le terminal.

**Une note sur `pyproject.toml`**, que tu croiseras aussi parfois : c'est l'évolution plus moderne du même
besoin, qui va plus loin — un seul fichier qui regroupe à la fois la liste des dépendances, des informations sur
le projet (son nom, sa version, son auteur), et la configuration de certains outils annexes (un formateur de
code, par exemple). Des outils comme `poetry` ou `uv` s'appuient sur ce fichier. Pour ce parcours,
`requirements.txt` est largement suffisant ; retiens simplement que `pyproject.toml` répond au même besoin de
base, en plus complet.

## 4. `src/` : pourquoi le code ne reste pas directement à la racine du projet

Tu rencontreras deux façons différentes d'organiser un projet Python. Regardons-les côte à côte :

```
SANS src/                       AVEC src/
projet_a/                       projet_b/
  mon_package/                    src/
    __init__.py                     mon_package/
    module.py                         __init__.py
  tests/                               module.py
  requirements.txt                 tests/
                                   requirements.txt
```

La disposition de droite, avec un dossier `src/` qui enveloppe le vrai code, est aujourd'hui la convention
recommandée par la communauté Python. Pour comprendre pourquoi, il faut d'abord comprendre un détail concret du
fonctionnement de Python.

**Quand tu lances un script Python depuis un dossier**, Python ajoute automatiquement ce dossier à la liste des
endroits où il va chercher les choses à importer (cette liste s'appelle le `PYTHONPATH`). Sans `src/`, ça veut
dire que le dossier racine de ton projet (`projet_a/`) devient lui-même automatiquement "importable", juste
parce que tu as lancé un script depuis là.

**Pourquoi c'est un problème caché.** Imagine que ton package `mon_package` a en réalité un petit défaut
d'installation (par exemple, il manque un fichier nécessaire pour qu'il fonctionne correctement une fois
vraiment installé ailleurs, sur un autre ordinateur). Si tu testes ton code *depuis la racine du projet
`projet_a/`*, tout fonctionnera quand même, parce que Python trouve `mon_package` directement là, sans jamais
passer par une vraie installation. Le défaut reste invisible en local, et tu ne le découvres que plus tard,
quand quelqu'un d'autre essaie d'installer ton package pour de vrai ailleurs, et que ça échoue.

**Avec `src/`**, ce problème disparaît : ton code n'est plus accessible "par accident" juste en étant dans le
bon dossier. Pour l'utiliser, il faut **vraiment l'installer comme un package** (avec `pip install -e .`, par
exemple, une commande qu'on ne détaille pas ici mais que tu croiseras). Si ça fonctionne en local avec `src/`,
c'est que ça fonctionnerait vraiment aussi ailleurs — le test est honnête.

**Le lien avec ce que tu connais déjà** : c'est exactement le même esprit que le dossier `src/` que tu as déjà
dans tes propres TD Java de 1A et 2A (avec un dossier `bin/` à côté pour le code compilé) — séparer clairement
"le code source" du reste (configuration, données, sorties générées).

## 5. `lib/` : ce que c'est vraiment, et pourquoi tu le connais déjà

Regarde ton propre dossier `Java example 1A/LIB/` : tu y trouveras des fichiers `.jar` (`TC.jar`, `MacLib.jar`).
Ce sont des bibliothèques externes, fournies directement sous cette forme compilée, posées à côté de ton code,
parce que ton cours de l'époque ne t'avait pas encore appris à utiliser un vrai gestionnaire de dépendances
automatique (comme Maven, l'équivalent Java de ce qu'on vient de voir avec `pip`).

En Python, un dossier `lib/` à la racine d'un repo GitLab a en général l'un de ces deux rôles, et il faut
regarder son contenu pour savoir lequel :

- **Le plus courant aujourd'hui** : un endroit pour du **code interne**, écrit par l'équipe elle-même et partagé
  entre plusieurs scripts ou services du même repo — pas une vraie bibliothèque externe téléchargée, juste une
  façon de séparer "le code réutilisable par plusieurs parties du projet" du "code qui est le point d'entrée
  principal".
- **Plus rare** : des dépendances externes **copiées directement** dans le repo plutôt qu'installées via `pip`
  (on appelle ça du *vendoring*), pour des cas particuliers — par exemple, un environnement de déploiement sans
  accès à internet, ou une bibliothèque qu'on a dû modifier soi-même à la main.

Avec `pip` et `requirements.txt`, ce deuxième cas est devenu rare : la convention moderne pour "du code interne
partagé" est plutôt un sous-dossier à l'intérieur de `src/` qu'un `lib/` séparé à la racine. Si tu vois `lib/`
dans un repo de ta boîte, ouvre-le et regarde ce qu'il contient avant de supposer lequel des deux cas s'applique
— c'est souvent une question d'histoire du projet plus que de règle stricte.

## 6. `.gitignore` : dire à Git ce qu'il ne doit jamais suivre

**Petit rappel de ce que fait Git**, pour bien poser le problème : Git garde un historique complet de tous les
fichiers que tu lui dis de suivre ("committer"). Chaque fichier ajouté reste dans cet historique pour toujours,
même si tu le supprimes plus tard — l'historique, lui, garde la trace de ce qu'il contenait.

**Trois catégories de fichiers n'ont jamais intérêt à être suivies par Git :**

1. **Ce qui est reconstruit automatiquement.** Le dossier `.venv` qu'on a créé en section 2 en est l'exemple
   parfait : il peut peser plusieurs centaines de mégaoctets, et n'importe qui peut le recréer à l'identique en
   deux commandes (`python -m venv .venv` puis `pip install -r requirements.txt`). Le committer gonflerait le
   repo pour absolument aucun bénéfice.
2. **Ce qui est spécifique à ta machine personnelle**, sans intérêt pour quelqu'un d'autre qui clonerait le
   projet (les réglages propres à ton éditeur de texte, par exemple).
3. **Les secrets.** C'est la catégorie la plus importante, détaillée dans la section suivante — et on vient d'en
   avoir un exemple très concret et bien réel juste avant ce notebook.

**Le fichier `.gitignore`** liste simplement, une ligne par motif, ce que Git doit ignorer et ne jamais proposer
de suivre :

```
__pycache__/
*.pyc
.venv/
.env
.ipynb_checkpoints/
```

Le `.gitignore` de ce dépôt (`cs-mastery-path/.gitignore`) contient déjà exactement ça — tu peux aller le
consulter directement.

## 7. `.env` : séparer les secrets du code — et une histoire vraie pour bien comprendre pourquoi

**Le problème de fond.** Un mot de passe de base de données, une clé d'accès à un service externe, une adresse
qui change entre ton ordinateur de développement et le serveur de production réel : ces valeurs ne doivent
**jamais** être écrites directement, en dur, dans ton code source. Pour deux raisons : si ce code est suivi par
Git (et il l'est presque toujours), le secret se retrouve dans l'historique, potentiellement visible par
d'autres personnes, pour toujours ; et si tu changes d'environnement (développement, puis production), il
faudrait modifier le code lui-même à chaque fois.

**Tu as, en fait, déjà vécu une version très concrète de ce danger pendant ce parcours.** Rappelle-toi : quand
on essayait de connecter Colab à GitHub, un message d'erreur contenait, caché dans une suite de caractères
qui ressemblait à du charabia, un vrai jeton d'accès (un "token") à ton compte GitHub. Ce jeton jouait exactement
le même rôle qu'un mot de passe : toute personne qui l'aurait récupéré aurait pu agir à ta place sur GitHub,
sans connaître ton vrai mot de passe. La bonne réaction, à ce moment précis, a été de le **révoquer**
immédiatement (l'invalider pour qu'il ne serve plus à rien) dès qu'on a compris qu'il était exposé. C'est
exactement le même risque, à plus petite échelle et de façon volontaire cette fois, qui se produirait si un
secret était laissé en clair dans du code committé sur Git : n'importe qui ayant accès au repo (ou à son
historique, même après coup) pourrait le récupérer et l'utiliser.

**La convention pour éviter ça.** On crée un fichier appelé `.env` à la racine du projet (le point devant le nom
signifie, par convention Unix, "fichier normalement caché"), qui contient des paires `CLE=valeur` :

```
DATABASE_URL=postgresql://localhost:5432/madb
API_KEY=sk-abc123...
```

Ce fichier `.env` est systématiquement ajouté au `.gitignore` (vu en section 6) : il ne doit **jamais** être
committé.

**Et dans le code**, on ne lit jamais la valeur directement : on utilise une petite bibliothèque, par exemple
`python-dotenv`, qui va lire ce fichier `.env` au démarrage du programme et rendre ses valeurs disponibles :

```python
from dotenv import load_dotenv
import os

load_dotenv()                       # lit le fichier .env et charge ses valeurs
api_key = os.environ["API_KEY"]     # jamais la vraie valeur écrite en dur dans ce fichier .py
```

**Un détail pratique utile** : comme `.env` n'est jamais committé, quelqu'un qui clone le projet pour la
première fois ne sait pas quelles clés il est censé renseigner. C'est pour ça qu'on ajoute souvent un deuxième
fichier, `.env.example`, qui **lui, est committé normalement**, et qui liste les mêmes clés mais avec des
valeurs factices :

```
DATABASE_URL=remplace_par_ta_propre_url
API_KEY=remplace_par_ta_propre_cle
```

## 8. Vue d'ensemble

En assemblant tout ce qu'on vient de construire, un projet Python bien structuré ressemble à ceci :

```
mon_projet/
├── src/
│   └── mon_projet/
│       ├── __init__.py
│       └── core.py
├── tests/
│   └── test_core.py
├── .env.example        <- committé : montre quelles clés sont attendues, sans leurs vraies valeurs
├── .env                <- jamais committé : les vraies valeurs secrètes, sur ta machine seulement
├── .gitignore          <- liste .venv/, .env, __pycache__/, etc.
├── requirements.txt    <- la liste exacte des bibliothèques nécessaires
└── README.md
```

Ce dépôt `cs-mastery-path` suit une version volontairement allégée de cette logique (`requirements.txt`,
`.gitignore`, `README.md` à la racine) — allégée parce qu'ici, le "code" ce sont des notebooks pédagogiques
destinés à être lus et exécutés directement, pas un package qu'on installe ailleurs.

## 9. À toi — exercice guidé

Pas dans ce notebook, mais directement sur ta machine (ou dans le terminal intégré de Colab). Crée un dossier
`mon_premier_repo/` et structure-le en suivant ces étapes, dans l'ordre, en cochant chacune une fois faite :

1. [ ] Crée le dossier, puis initialise un dépôt Git à l'intérieur (`git init`).
2. [ ] Crée un environnement virtuel nommé `.venv` (section 2), et active-le. Vérifie que le nom de
   l'environnement apparaît bien dans ton terminal une fois activé.
3. [ ] Crée les fichiers `src/mon_premier_repo/__init__.py` (vide) et `src/mon_premier_repo/core.py`, avec une
   fonction `bonjour(nom: str) -> str` qui renvoie `f"Bonjour {nom}"`.
4. [ ] Crée un fichier `requirements.txt` (même vide pour l'instant, juste le fichier lui-même).
5. [ ] Crée un `.gitignore` contenant au moins `__pycache__/`, `.venv/`, `.env`.
6. [ ] Crée un fichier `.env` avec une ligne `NOM_PAR_DEFAUT=Youssef`, puis un `.env.example` avec
   `NOM_PAR_DEFAUT=remplace_par_ton_nom`.
7. [ ] Tape `git status`. Vérifie bien que `.venv/` et `.env` **n'apparaissent pas** dans la liste des fichiers
   proposés à committer. S'ils apparaissent, ton `.gitignore` a un problème — corrige-le avant de continuer, ne
   passe pas à l'étape suivante avec ce problème non résolu.
8. [ ] Fais un premier commit propre (`git add` puis `git commit -m "..."`), et vérifie avec `git log` qu'il
   apparaît bien.

Une fois que tu auras fait ça une fois entièrement à la main, tu comprendras vraiment ce que fait un générateur
de projet automatique (comme `cookiecutter`) quand tu en utiliseras un plus tard : il se contente d'automatiser
exactement ces mêmes étapes, dans le même ordre.

---

**Brique 0 terminée.** Relis-la une dernière fois en entier si besoin, coche-la dans
[ROADMAP.md](../ROADMAP.md), puis dis-moi quand tu es prêt pour la Brique 1 (rappel Python de base).